In [ ]:
import sys
import os 
from SYK_fft import *
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from ConformalAnalytical import *
from SUP_iterator_submission import * 
from scipy.signal import find_peaks

In [ ]:
DUMP = True
M = int(2**16) #number of points in the grid
T = 2**12 #upper cut-off for the time
err = 1e-5 #error treshold for convergence check


omega,t  = RealGridMaker(M,T)
dw = omega[2] - omega[1]
dt = t[2] - t[1]

print("dw = ", dw)
print("dt = ", dt)

delta = 0.420374134464041
ITERMAX = 55000
global beta

mu = 0.0
g = .5
r = 1.
kappa = 1.
eta = dw*20.1 #may influence Tc

alpha=0.
betastart = 35
beta = betastart
betatarget = 45
beta_step = 1

grid = [M,omega,t]
pars = [g,mu,r]




In [ ]:
#load a normal solution near the transition temperature:
GRomega, DRomega = np.load('M16T12beta33_0g0_5r1_0.npy')
FRomega =0.1*(1+1j)*np.ones_like(GRomega) #initial guess for the anomalous Green's function
FRomega[M:] = FRomega[M:]*-1 #enforce odd

In [ ]:
#running this will take a while (>1h). Around beta 43 it will converge on a gapped SC solution. 
while beta < betatarget:
    print(beta)
    GRomega, DRomega, FRomega, INFO = GF_RE_SUP_iterator(GRomega,DRomega,FRomega,grid,pars,beta, alpha, err=err,ITERMAX=ITERMAX,eta = eta,verbose=True,diffcheck=False)
    itern, diff = INFO
    
    if DUMP == True :
        savefile = 'trial5_ysykSUP_' +'M' + str(int(np.log2(M))) + 'T' + str(int(np.log2(T))) 
        savefile += 'beta' + str((round(beta*100))/100.) 
        savefile += 'g' + str(g) + 'r' + str(r) + 'eta'+str(round(eta/dw,3)) + 'alpha' + str(round(alpha,3))
        savefile = savefile.replace('.','_') 
        savefile +=  '.npy' 
        print('saveas', savefile) 
        np.save(savefile, np.array([GRomega,DRomega,FRomega]))
    
    fig, ax = plt.subplots(2)
    fig.suptitle('Iteration = ' + str(itern) + ', beta = ' + str(beta))
    ax[0].plot(omega, -1*np.imag(GRomega), label = 'Im(GR)')
    ax[1].plot(omega, -1*np.imag(FRomega), label = 'Im(DR)')
    ax[0].set_xlabel(r'$\omega$')
    ax[0].set_ylabel(r'$-Im G^R(\omega)$')
    ax[0].set_xlim(0,1)
    ax[1].set_xlabel(r'$\omega$')
    ax[1].set_ylabel(r'$-Im F^R(\omega)$')
    ax[1].set_xlim(0,1)
   

    #plot if you want
    fig.set_size_inches(6,6)
    fig.tight_layout()
    plt.show()

    print(f'End F[0] = {FRomega[M+1]}')
    print("##### Finished beta = ", beta," in ", itern, " iterations with diff = ", diff, " ############")

    beta += beta_step